# Kalp Sesi Sınıflandırma — Eğitim Notebook'u

**MobileNetV2 Transfer Learning ile PCG Sınıflandırması**

Bu notebook Aşama 7–10'u çalıştırır:
- Aşama 7: Model mimarisi (MobileNetV2)
- Aşama 8: İki aşamalı eğitim (T4 GPU)
- Aşama 9: Test seti değerlendirmesi
- Aşama 10: TFLite dönüşümü ve doğrulama

**Gereksinimler:** Google Colab + T4 GPU, veri Google Drive'da

## 0. Ortam Kurulumu

In [ ]:
# GPU kontrolü
!nvidia-smi
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Proje dosyalarını Drive'dan kopyala
DRIVE_PROJECT = '/content/drive/MyDrive/heart-sound-classifier'

# 1. ZIP'i yerel diske kopyala (Drive'dan dogrudan unzip cok yavas)
print("ZIP kopyalaniyor...")
!cp {DRIVE_PROJECT}/data.zip /content/data.zip
print("ZIP kopyalandi, aciliyor...")

# 2. Yerel diskte ac (-o = overwrite sormadan)
!unzip -q -o /content/data.zip -d /content/data
!rm /content/data.zip
print("ZIP acildi!")

# 3. Kaynak kodu ve config kopyala
!cp -r {DRIVE_PROJECT}/src /content/src
!cp -r {DRIVE_PROJECT}/configs /content/data/configs
!cp -r {DRIVE_PROJECT}/splits /content/data/splits
print("Dosyalar kopyalandi!")

# 4. CSV'lerdeki yol prefix'ini duzelt (data/mel/ -> mel/)
from pathlib import Path
for split in ['train', 'val', 'test']:
    csv_path = Path(f'/content/data/splits/{split}.csv')
    content = csv_path.read_text()
    if 'data/mel/' in content:
        csv_path.write_text(content.replace('data/mel/', 'mel/'))
        print(f'{split}.csv yol duzeltildi')

import sys
sys.path.insert(0, '/content')
print("Tum dosyalar hazir!")

In [ ]:
# Eksik paketleri kur
!pip install -q pyyaml scikit-learn seaborn

In [ ]:
# Yolları tanımla
from pathlib import Path

DATA_DIR = Path('/content/data')
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Drive'a checkpoint kaydetmek için
DRIVE_OUTPUT = Path(DRIVE_PROJECT) / 'output'
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f'Veri dizini: {DATA_DIR}')
print(f'Çıktı dizini: {OUTPUT_DIR}')
print(f'Drive çıktı: {DRIVE_OUTPUT}')

In [ ]:
# Veri dosyalarını doğrula
import os
mel_count = len(list((DATA_DIR / 'mel').glob('*.npy')))
print(f'Mel dosya sayısı: {mel_count}')

for split in ['train', 'val', 'test']:
    with open(DATA_DIR / 'splits' / f'{split}.csv') as f:
        lines = sum(1 for _ in f) - 1
    print(f'{split}: {lines} örnek')

## Aşama 7 — Model Mimarisi

In [ ]:
from src.models.build_model import build_stage1_model, unfreeze_for_stage2, get_model_summary

model = build_stage1_model()
model.summary()

In [ ]:
# Model mimarisi görseli
import os
os.makedirs(str(OUTPUT_DIR / 'figures'), exist_ok=True)
tf.keras.utils.plot_model(
    model, to_file=str(OUTPUT_DIR / 'figures' / 'fig_model_architecture.png'),
    show_shapes=True, show_layer_names=True, dpi=150
)

## Aşama 8 — İki Aşamalı Eğitim

In [ ]:
import yaml
import json
import numpy as np
import csv
from src.training.train import create_dataset, load_split_data, balanced_class_weights

# Config yükle
# v2 (deployed): 'config.yaml'  |  v4 without BUET and Yaseen (advisor's review K2): 'config_v4_no_buet_yaseen.yaml'
CONFIG_FILE = 'config.yaml'
with open(DATA_DIR / 'configs' / CONFIG_FILE) as f:
    config = yaml.safe_load(f)

# Sınıf ağırlıkları
with open(DATA_DIR / 'configs' / 'class_weights.json') as f:
    class_weights = {int(k): v for k, v in json.load(f).items()}

EXCLUDE = config['training'].get('exclude_datasets') or []
print(f'Config: {CONFIG_FILE}, hariç tutulan veri setleri: {EXCLUDE or "yok"}')
print(f'Sınıf ağırlıkları: {class_weights}')

train_cfg = config['training']
BATCH_SIZE = train_cfg['batch_size']

In [ ]:
# Veri setlerini yükle
train_paths, train_labels = load_split_data(DATA_DIR / 'splits' / 'train.csv', DATA_DIR, EXCLUDE)
val_paths, val_labels = load_split_data(DATA_DIR / 'splits' / 'val.csv', DATA_DIR, EXCLUDE)
if EXCLUDE:  # stored class weights were computed on the full training split
    class_weights = balanced_class_weights(train_labels)
    print(f'Sınıf ağırlıkları (yeniden hesaplandı): {class_weights}')

AUG = config['training'].get('augmentation', {})
print('Veri artırma:', AUG if AUG.get('enabled') else 'kapalı')
train_ds = create_dataset(train_paths, train_labels, BATCH_SIZE, shuffle=True,
                          augment=AUG.get('enabled', False), aug_cfg=AUG)
val_ds = create_dataset(val_paths, val_labels, BATCH_SIZE)

print(f'Eğitim: {len(train_paths)} örnek')
print(f'Doğrulama: {len(val_paths)} örnek')

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)

# Çıktı dizinleri
models_dir = OUTPUT_DIR / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
logs_dir = OUTPUT_DIR / 'logs'
logs_dir.mkdir(parents=True, exist_ok=True)
figures_dir = OUTPUT_DIR / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# ========== AŞAMA 1: Özellik Çıkarma (base donmuş) ==========
print('='*50)
print('AŞAMA 1: Base donmuş, lr=1e-3, 10 epoch')
print('='*50)

model = build_stage1_model()
model.compile(
    optimizer=Adam(learning_rate=train_cfg['stage1']['learning_rate']),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')],
)

stage1_callbacks = [
    ModelCheckpoint(
        str(models_dir / 'best_stage1.keras'),
        monitor='val_loss', save_best_only=True, verbose=1,
    ),
    # Drive'a da kaydet (Colab kopma koruması)
    ModelCheckpoint(
        str(DRIVE_OUTPUT / 'best_stage1.keras'),
        monitor='val_loss', save_best_only=True, verbose=0,
    ),
    CSVLogger(str(logs_dir / 'stage1_log.csv')),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=train_cfg['stage1']['epochs'],
    class_weight=class_weights,
    callbacks=stage1_callbacks,
    verbose=1,
)

In [ ]:
# ========== AŞAMA 2: İnce Ayar (son 4 blok açık) ==========
print('='*50)
print('AŞAMA 2: Son 4 blok açık, lr=1e-5, 30 epoch')
print('='*50)

model.load_weights(str(models_dir / 'best_stage1.keras'))
model = unfreeze_for_stage2(model, n_blocks=train_cfg['stage2']['unfreeze_blocks'])

model.compile(
    optimizer=Adam(learning_rate=train_cfg['stage2']['learning_rate']),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')],
)

stage2_callbacks = [
    ModelCheckpoint(
        str(models_dir / 'best_model.keras'),
        monitor='val_loss', save_best_only=True, verbose=1,
    ),
    ModelCheckpoint(
        str(DRIVE_OUTPUT / 'best_model.keras'),
        monitor='val_loss', save_best_only=True, verbose=0,
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=train_cfg['early_stopping_patience'],
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=train_cfg['reduce_lr_factor'],
        patience=train_cfg['reduce_lr_patience'],
        min_lr=1e-7,
        verbose=1,
    ),
    CSVLogger(str(logs_dir / 'stage2_log.csv')),
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=train_cfg['stage2']['epochs'],
    class_weight=class_weights,
    callbacks=stage2_callbacks,
    verbose=1,
)

In [ ]:
# Final modeli kaydet
model.save(str(models_dir / 'final_model.keras'))
model.save(str(DRIVE_OUTPUT / 'final_model.keras'))
print('Final model kaydedildi.')

In [ ]:
# Eğitim eğrilerini çiz
from src.training.train import _plot_training_curves
_plot_training_curves(history1, history2, figures_dir)

## Aşama 9 — Test Seti Değerlendirmesi

In [ ]:
from src.evaluation.evaluate import evaluate

metrics = evaluate(
    data_dir=DATA_DIR,
    model_path=models_dir / 'best_model.keras',
    output_dir=OUTPUT_DIR,
)

In [ ]:
# Sonuçları göster
import pandas as pd

metrics_display = {k: f'{v:.4f}' for k, v in metrics.items() 
                   if isinstance(v, float)}
pd.DataFrame([metrics_display], index=['Test Seti']).T

## Aşama 10 — TFLite Dönüşümü

In [ ]:
from src.deployment.convert_tflite import convert_to_tflite, consistency_test

tflite_path = models_dir / 'heart_sound_classifier_fp16.tflite'
convert_to_tflite(models_dir / 'best_model.keras', tflite_path)

In [ ]:
# Tutarlılık testi
comparison = consistency_test(
    keras_model_path=models_dir / 'best_model.keras',
    tflite_path=tflite_path,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
)

In [ ]:
# TFLite modelini Drive'a kopyala
import shutil
shutil.copy(tflite_path, DRIVE_OUTPUT / 'heart_sound_classifier_fp16.tflite')
print(f'TFLite model Drive\'a kopyalandı: {DRIVE_OUTPUT}')

## Tüm Çıktıları Drive'a Kaydet

In [ ]:
# Tüm figürleri ve sonuçları Drive'a kopyala
for subdir in ['figures', 'results', 'logs']:
    src = OUTPUT_DIR / subdir
    dst = DRIVE_OUTPUT / subdir
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Kopyalandı: {subdir}/')

print('\nTüm çıktılar Drive\'a kaydedildi!')
print(f'Konum: {DRIVE_OUTPUT}')